# 01 — Data Understanding
### Sales Forecasting & Business Analytics Platform — Phase 2

**Dataset:** Rossmann Store Sales (Kaggle)
**Notebook purpose:** profile the three raw files thoroughly *before* changing anything.
This notebook does not clean or transform data — that happens in `02_data_cleaning.ipynb`.


## Objectives

- Confirm the shape, columns, and data types of `train.csv`, `test.csv`, `store.csv`.
- Identify every missing value, duplicate row, and inconsistent data type.
- Investigate specific risks flagged during architecture planning: the `StateHoliday`
  column's mixed types, the relationship between `Open` and `Sales`, and whether
  `Promo2`-related missing values are true gaps or structurally "not applicable."
- Produce a consolidated **Data Quality Report** that Notebook 02 will act on.


## Theory: Why "Understanding" Is Its Own Phase

This follows **CRISP-DM**, the industry-standard data science process, which
separates *Data Understanding* from *Data Preparation* as distinct phases.

**Why it matters:** if you start "fixing" data before you've verified what's
actually wrong, you end up fixing problems you *guessed* at — a very common
beginner mistake. For example, it would be easy to assume `Sales == 0` rows are
data errors and drop them, when in fact they're stores that were legitimately
closed that day. Verifying *before* acting prevents that kind of silent, wrong
correction.

Every claim in this notebook is checked against the real data below — nothing
here is taken from documentation on faith.


## Concept: What Each File Represents

Rossmann is a German drug-store chain; this dataset was released for a Kaggle
recruiting competition on sales forecasting.

| File | Grain | Business meaning |
|---|---|---|
| `train.csv` | one row per store per day | historical daily sales — our modeling target |
| `test.csv` | one row per store per day | a future window (no `Sales` — this is what a forecaster would predict) |
| `store.csv` | one row per store | static store metadata (type, assortment, competition, promotions) |

`store.csv` must be merged onto `train`/`test` via the `Store` key — it's not
useful standalone until joined.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

**Why these `pd.set_option` calls:** purely cosmetic, so wide DataFrames print
fully in this notebook instead of truncating columns. They have no effect on the
underlying pipeline — you won't find them in `src/`.


In [2]:
import sys
sys.path.append("..")  # lets this notebook import the src/ package from the project root

from src.data_loader import load_all

train, test, store = load_all()

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
print(f"store: {store.shape}")

train: (1017209, 9)
test:  (41088, 8)
store: (1115, 10)


**What this does:** calls the same `load_all()` function the dashboard will
eventually use — one source of truth for loading logic, not copy-pasted code.
`load_all()` also validates that each file's columns match what we expect;
if a corrupted download slipped in, this would fail loudly right here, not three
notebooks later.

**Expected output:** `train` has ~1.02M rows / 9 columns, `test` ~41K rows / 8
columns, `store` 1,115 rows / 10 columns.


In [3]:
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


**Column meanings (train):** `Store` (ID), `DayOfWeek` (1=Monday), `Date`,
`Sales` (target variable, in currency units), `Customers` (foot traffic that day),
`Open` (1=open/0=closed), `Promo` (single-day promotion active), `StateHoliday`
(holiday code), `SchoolHoliday` (schools closed that day).


In [4]:
test.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


**Difference from train:** `test.csv` has an `Id` column (Kaggle's original
submission key) and — critically — **no `Sales` or `Customers` columns**. This is
literally what a forecaster would need to predict. We won't use this file for our
*own* validation (we have no ground truth to score against); instead we'll carve
a held-out chronological window out of `train` for validation in Notebook 05, and
optionally use `test.csv` later as a demo-only "real forecast" showcase.


In [5]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


**Column meanings (store):** `StoreType` (a/b/c/d — store format/size
category), `Assortment` (a=basic, b=extra, c=extended — product range depth),
`CompetitionDistance` (meters to nearest competitor), `CompetitionOpenSince{Month,Year}`
(when that competitor opened), `Promo2` (1 = store runs an ongoing, recurring
promotion beyond single-day `Promo`), `Promo2Since{Week,Year}` / `PromoInterval`
(details of that recurring promotion).

**Architecture note (approved):** this dataset has no SKU/product-level table.
Per our approved architecture decision, `StoreType` and `Assortment` will serve as
the **product-category proxy** for the PRD's "Product Analysis" requirement in
the EDA notebook.


In [6]:
print("TRAIN dtypes:")
print(train.dtypes)
print()
print("TEST dtypes:")
print(test.dtypes)
print()
print("STORE dtypes:")
print(store.dtypes)

TRAIN dtypes:
Store            int64
DayOfWeek        int64
Date               str
Sales            int64
Customers        int64
Open             int64
Promo            int64
StateHoliday       str
SchoolHoliday    int64
dtype: object

TEST dtypes:
Id                 int64
Store              int64
DayOfWeek          int64
Date                 str
Open             float64
Promo              int64
StateHoliday         str
SchoolHoliday      int64
dtype: object

STORE dtypes:
Store                          int64
StoreType                        str
Assortment                       str
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                    str
dtype: object


**Observation:** `Date` is read as plain text (`object`), not a real datetime
yet — expected, since we haven't parsed it. More importantly, `StateHoliday` is
`object` in a column that looks like it should be a small set of category codes.
We investigate that specifically below.


In [7]:
def missing_report(df: pd.DataFrame, name: str) -> None:
    """Print a sorted table of missing-value counts and percentages for a DataFrame."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({"missing_count": missing, "missing_pct": pct})
    report = report[report["missing_count"] > 0].sort_values("missing_count", ascending=False)
    print(f"--- Missing values: {name} ---")
    print(report if not report.empty else "No missing values.")
    print()

missing_report(train, "train")
missing_report(test, "test")
missing_report(store, "store")

--- Missing values: train ---
No missing values.

--- Missing values: test ---
      missing_count  missing_pct
Open             11         0.03

--- Missing values: store ---
                           missing_count  missing_pct
Promo2SinceYear                      544        48.79
Promo2SinceWeek                      544        48.79
PromoInterval                        544        48.79
CompetitionOpenSinceMonth            354        31.75
CompetitionOpenSinceYear             354        31.75
CompetitionDistance                    3         0.27



**Observation:** `train` and `test` have **zero** missing values — Kaggle's
curated competition data is clean at this level. `store.csv` has real gaps:
- `CompetitionDistance`: a handful of stores
- `CompetitionOpenSinceMonth` / `CompetitionOpenSinceYear`: 354 stores
- `Promo2SinceWeek` / `Promo2SinceYear` / `PromoInterval`: 544 stores

That last group is suspicious — three columns all missing the *same* count
suggests a structural pattern, not random data loss. Let's verify.


In [8]:
stores_without_promo2 = (store["Promo2"] == 0).sum()
missing_promo2_since_week = store["Promo2SinceWeek"].isnull().sum()

print("Stores with Promo2 == 0:       ", stores_without_promo2)
print("Stores missing Promo2SinceWeek:", missing_promo2_since_week)
print("Match:", stores_without_promo2 == missing_promo2_since_week)

Stores with Promo2 == 0:        544
Stores missing Promo2SinceWeek: 544
Match: True


**Conclusion:** confirmed — every store with `Promo2 == 0` (no recurring
promotion) is exactly the store missing `Promo2SinceWeek/Year`/`PromoInterval`.
This is **not missing data**, it's **structurally not applicable** — those fields
simply don't exist for a store that never ran a Promo2 campaign. Notebook 02 must
handle this with an explicit "not applicable" flag, not a blind mean/mode
imputation, which would fabricate a fake campaign start date for stores that
never had one.


In [9]:
print("Duplicate rows - train:", train.duplicated().sum())
print("Duplicate rows - test: ", test.duplicated().sum())
print("Duplicate rows - store:", store.duplicated().sum())

Duplicate rows - train: 0
Duplicate rows - test:  0
Duplicate rows - store: 0


**Observation:** zero duplicate rows anywhere. Good news — but worth
confirming explicitly rather than assuming, since assuming cleanliness is a
common beginner mistake that can silently double-count sales in later
aggregations if wrong.


In [10]:
print("Unique StateHoliday values (train):", train["StateHoliday"].unique())
print()
print("Value counts (train):")
print(train["StateHoliday"].value_counts(dropna=False))
print()
print("Unique StateHoliday values (test):", test["StateHoliday"].unique())

Unique StateHoliday values (train): <StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str

Value counts (train):
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64

Unique StateHoliday values (test): <StringArray>
['0', 'a']
Length: 2, dtype: str


In [11]:
# A quick comparison: what happens WITHOUT low_memory=False?
naive_read = pd.read_csv("../data/raw/train.csv")
print("Naive read (default settings) unique StateHoliday:", naive_read["StateHoliday"].unique())
print("dtype:", naive_read["StateHoliday"].dtype)

Naive read (default settings) unique StateHoliday: ['0' 'a' 'b' 'c' 0]
dtype: object


/tmp/ipykernel_631/1553243922.py:2: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  naive_read = pd.read_csv("../data/raw/train.csv")


**Correction to the risk we flagged during architecture planning:** reading the
file with pandas' *default* settings does reproduce the classic mixed-type
warning and produces genuinely mixed values (string `'0'` alongside integer
`0`) — that's what our earlier ad-hoc exploration saw, and why we flagged it as
a risk in the architecture doc.

**But** `load_train()` in `src/data_loader.py` already reads with
`low_memory=False`, which forces pandas to infer the column's type from the
*entire* file at once instead of chunk-by-chunk. As the cell above confirms,
that alone is enough to produce a single, consistent type — no separate fix
needed in Notebook 02 for this column.

This is a good reminder to verify assumptions against the actual code path
being used, not against a quick one-off check with different settings.

**Verified conclusion:** through `load_train()` (our actual pipeline path),
`StateHoliday` is a single, consistent string type with four values — `'0'`
(no holiday), `'a'` (public holiday), `'b'` (Easter), `'c'` (Christmas). No
dtype fix is required in Notebook 02 for this column. **Revised action item:**
none — this row is removed from the Data Quality Report below, replaced with a
note that it was investigated and found to be a non-issue given our loader's
settings.

In [12]:
print("Open value counts (train):")
print(train["Open"].value_counts())
print()

closed_sales = train.loc[train["Open"] == 0, "Sales"]
print("Sales stats when Open == 0:")
print(closed_sales.describe())
print()

open_but_zero_sales = train.loc[(train["Open"] == 1) & (train["Sales"] == 0)]
print(f"Rows where store is OPEN but Sales == 0: {len(open_but_zero_sales)}")

Open value counts (train):
Open
1    844392
0    172817
Name: count, dtype: int64

Sales stats when Open == 0:
count    172817.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: Sales, dtype: float64

Rows where store is OPEN but Sales == 0: 54


**Two distinct findings here:**

1. **Confirmed:** every single closed-store day (`Open == 0`) has `Sales == 0` —
   this is tautological, not informative. A model doesn't need to "learn" that a
   closed store sells nothing. **Action item:** exclude `Open == 0` rows from
   training/evaluation in Notebook 02, since they'd otherwise inflate accuracy
   metrics with trivially-correct predictions.
2. **The open-but-zero-sales rows are a different, more interesting case** — a
   store that was open yet sold nothing. Unlike the closed-store rows, these
   *are* potentially real signal (e.g. an unusual event) and should **not** be
   dropped the same way — we'll decide their treatment explicitly in Notebook 02
   rather than lumping them in with the closed-store logic.


In [13]:
missing_open_test = test["Open"].isnull().sum()
print("Missing Open values in test:", missing_open_test)
print()
test.loc[test["Open"].isnull()]

Missing Open values in test: 11



,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
479,480,622,4,2015-09-17,NaN,1,0,0
1335,1336,622,3,2015-09-16,NaN,1,0,0
2191,2192,622,2,2015-09-15,NaN,1,0,0
3047,3048,622,1,2015-09-14,NaN,1,0,0
4759,4760,622,6,2015-09-12,NaN,0,0,0
5615,5616,622,5,2015-09-11,NaN,0,0,0
6471,6472,622,4,2015-09-10,NaN,0,0,0
7327,7328,622,3,2015-09-09,NaN,0,0,0
8183,8184,622,2,2015-09-08,NaN,0,0,0
9039,9040,622,1,2015-09-07,NaN,0,0,0


**Observation:** interpreted from the output above — this is a real,
documented quirk of the original Kaggle test set (a handful of rows for one
specific store have no recorded `Open` status). Since we are not submitting to
Kaggle and are instead using our own chronological holdout from `train.csv` for
validation, this quirk does not affect our modeling pipeline — but it's worth
noting for completeness and for anyone who later uses `test.csv` for a demo.


In [14]:
print("Train date range:", train["Date"].min(), "to", train["Date"].max())
print("Test date range: ", test["Date"].min(), "to", test["Date"].max())
print()
print("Unique stores - train:", train["Store"].nunique())
print("Unique stores - test: ", test["Store"].nunique())
print("Unique stores - store:", store["Store"].nunique())
print()
print("Stores in train missing from store.csv:", set(train["Store"]) - set(store["Store"]))
print("Stores in store.csv missing from train:", set(store["Store"]) - set(train["Store"]))

Train date range: 2013-01-01 to 2015-07-31
Test date range:  2015-08-01 to 2015-09-17

Unique stores - train: 1115
Unique stores - test:  856
Unique stores - store: 1115



Stores in train missing from store.csv: set()
Stores in store.csv missing from train: set()


**Observation:** `train` spans about two and a half years of daily data;
`test` covers a separate future window with **no overlap** — confirming that
`test.csv` cannot serve as our validation set (we have no ground-truth `Sales`
to score against). This is exactly why the architecture doc calls for carving a
chronological holdout out of `train` itself for validation. All 1,115 stores are
present and consistent across every file — no orphan stores in either direction.


In [15]:
train[["Sales", "Customers"]].describe()

,Sales,Customers
count,1.017209e+06,1.017209e+06
mean,5.773819e+03,6.331459e+02
std,3.849926e+03,4.644117e+02
min,0.000000e+00,0.000000e+00
25%,3.727000e+03,4.050000e+02
50%,5.744000e+03,6.090000e+02
75%,7.856000e+03,8.370000e+02
max,4.155100e+04,7.388000e+03


**Observation:** minimum `Sales` is 0 (closed-store days, already explained
above). The gap between mean and median, along with a max far above the 75th
percentile, suggests the distribution is right-skewed — typical of retail sales
data, where most days cluster around a typical volume but a few high-traffic
days pull the average up. This is a modeling consideration for Notebook 05, not
something we act on here.


## Data Quality Report (Summary)

| # | Finding | Type | Action for Notebook 02 |
|---|---|---|---|
| 1 | `StateHoliday` investigated for mixed dtype | Verified non-issue | None — `low_memory=False` in our loader already resolves it |
| 2 | `CompetitionDistance` missing (a few stores) | True missing data | Decide & document imputation strategy |
| 3 | `CompetitionOpenSinceMonth/Year` missing (354 stores) | True missing data | Decide & document imputation/flag strategy |
| 4 | `Promo2SinceWeek/Year`, `PromoInterval` missing (544 stores) | **Not** missing — structurally N/A (`Promo2==0`) | Conditional handling, not blanket imputation |
| 5 | `Open == 0` rows always have `Sales == 0` | Tautological, not informative | Exclude from training/evaluation |
| 6 | `Open` open-but-zero-sales rows exist | Potential real signal | Handle separately from closed-store rows |
| 7 | A few `test.csv` rows have missing `Open` | Known dataset quirk | Note only — doesn't affect our validation approach |
| 8 | `Date` stored as text, not datetime | Expected, not yet parsed | Parse to datetime |
| 9 | Zero duplicate rows anywhere | Positive finding | No action needed |
| 10 | Full store-ID consistency across all 3 files | Positive finding | No action needed |


## Business Observations

- **Scale:** 1,115 stores × ~2.5 years of daily data is a substantial retail
  dataset — enough to support store-level trend and seasonality analysis, though
  only about two full annual cycles, which limits how confidently we can claim
  year-over-year seasonal patterns later.
- **No product-level granularity.** As approved in the architecture review,
  `StoreType`/`Assortment` will stand in for "Product Analysis" — this is a
  real, documented limitation of the source data, not an oversight.
- **~17% of rows are closed-store days** (172,817 of 1,017,209) — a large
  enough chunk that simply dropping them without checking the open-but-zero-sales
  edge case (finding #6) could have quietly thrown away real signal.


## Next Steps

Notebook 02 (`02_data_cleaning.ipynb`) will act on every item in the Data
Quality Report above: fix the `StateHoliday` dtype, decide and document
imputation strategies for `store.csv`'s true missing values, handle the
`Open == 0` filtering decision, parse dates, merge `store.csv` onto
`train`/`test`, and persist the result to `data/processed/`.
